# 🫁 Chest X-Ray AI Pipeline
## Production Training & Fine-Tuning — DenseNet-121 Classifier + Attention-UNet Segmenter

**Target:** >97% val accuracy on 5-class chest X-ray classification + binary lung segmentation  
**Output:** `custom_classifier.pth` and `custom_segmenter.pth` — backend-compatible weight files

---
### Notebook Structure
1. Environment Setup & GPU Check
2. Dataset Download & Structure
3. Preprocessing, Augmentation & DataLoaders
4. Model Architecture (Backend-Exact Definitions)
5. Class Imbalance Handling
6. Classifier Training Loop (DenseNet-121)
7. Segmenter Training Loop (Attention-UNet)
8. Evaluation, Metrics & Confusion Matrix
9. Weights Export & Verification
10. Inference Smoke Test

## 1. Environment Setup & GPU Check

In [ ]:
# ── Install / upgrade critical libraries ──────────────────────────────────────
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q albumentations==1.4.3 segmentation-models-pytorch timm opencv-python-headless
!pip install -q scikit-learn matplotlib seaborn tqdm kaggle gdown

import os, sys, json, random, shutil, zipfile, gc
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast

import torchvision
import torchvision.transforms as T
from torchvision import models

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score, accuracy_score
)

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── Device ─────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')

## 2. Configuration — All Hyperparameters in One Place

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  GLOBAL CONFIGURATION  — edit here, nowhere else
# ══════════════════════════════════════════════════════════════════════════════
CFG = {
    # ── Paths ──────────────────────────────────────────────────────────────────
    'data_root'          : '/content/data',
    'clf_data_dir'       : '/content/data/classifier',   # subfolders = class names
    'seg_img_dir'        : '/content/data/segmentation/images',
    'seg_mask_dir'       : '/content/data/segmentation/masks',
    'output_dir'         : '/content/weights',
    'clf_weights'        : '/content/weights/custom_classifier.pth',
    'seg_weights'        : '/content/weights/custom_segmenter.pth',

    # ── Classes (must mirror backend) ─────────────────────────────────────────
    'classes'            : ['Normal', 'Pneumonia', 'COVID-19', 'Tuberculosis', 'Lung Opacity'],
    'n_classes'          : 5,

    # ── Image sizes ────────────────────────────────────────────────────────────
    'clf_img_size'       : 224,
    'seg_img_size'       : 256,

    # ── ImageNet normalization (required by backend) ───────────────────────────
    'mean'               : [0.485, 0.456, 0.406],
    'std'                : [0.229, 0.224, 0.225],

    # ── Classifier training ────────────────────────────────────────────────────
    'clf_batch_size'     : 32,
    'clf_epochs'         : 50,
    'clf_lr'             : 1e-4,
    'clf_weight_decay'   : 1e-4,
    'clf_warmup_epochs'  : 5,
    'clf_patience'       : 10,         # early stopping patience
    'clf_label_smoothing': 0.05,
    'clf_mixup_alpha'    : 0.4,
    'clf_cutmix_alpha'   : 0.4,
    'clf_tta_n'          : 5,           # test-time augmentation repeats

    # ── Segmenter training ────────────────────────────────────────────────────
    'seg_batch_size'     : 16,
    'seg_epochs'         : 60,
    'seg_lr'             : 1e-4,
    'seg_weight_decay'   : 1e-5,
    'seg_warmup_epochs'  : 5,
    'seg_patience'       : 12,
    'seg_pos_weight'     : 2.0,        # up-weight positive mask pixels

    # ── Misc ──────────────────────────────────────────────────────────────────
    'val_split'          : 0.15,
    'test_split'         : 0.10,
    'num_workers'        : 2,
    'amp'                : True,        # automatic mixed precision
    'grad_clip'          : 1.0,
    'accum_steps'        : 2,           # gradient accumulation
}

os.makedirs(CFG['output_dir'], exist_ok=True)
os.makedirs(CFG['data_root'], exist_ok=True)
print('Config loaded ✓')
print(f"Classes : {CFG['classes']}")

## 3. Dataset Download

> **Options A–C below — uncomment the one that fits your data source.**  
> The pipeline expects a flat `classifier/` folder with one sub-folder per class,  
> and `segmentation/images` + `segmentation/masks` for the UNet.

In [ ]:
# ── Option A: Kaggle datasets (requires kaggle.json in /root/.kaggle/) ─────────
# Classifier data  — NIH + Kaggle COVID-19 CXR + TB + Opacity datasets
#
# !mkdir -p ~/.kaggle
# from google.colab import files
# uploaded = files.upload()          # upload your kaggle.json
# !cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
#
# !kaggle datasets download -d tawsifurrahman/covid19-radiography-database -p /content/raw --unzip
# !kaggle datasets download -d jtiptj/chest-xray-pneumoniacovid19tuberculosis -p /content/raw --unzip
#
# ── Option B: Google Drive (your own pre-split dataset) ───────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/chest_xray_data /content/data
#
# ── Option C: gdown (public GDrive share) ─────────────────────────────────────
# !gdown --id YOUR_FILE_ID -O /content/data.zip
# !unzip -q /content/data.zip -d /content/data

# ── Demo: Create synthetic folder structure so the rest of the notebook runs ──
#    Remove this block when using real data.
import shutil
from PIL import Image

DEMO_MODE = True   # ← Set False when using real data

if DEMO_MODE:
    print('⚠  DEMO MODE — generating synthetic data for pipeline testing')
    # Classifier: ~100 random images per class (imbalanced to stress-test)
    DEMO_COUNTS = {'Normal': 300, 'Pneumonia': 150, 'COVID-19': 80,
                   'Tuberculosis': 40, 'Lung Opacity': 60}
    for cls, n in DEMO_COUNTS.items():
        d = Path(CFG['clf_data_dir']) / cls
        d.mkdir(parents=True, exist_ok=True)
        for i in range(n):
            arr = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
            Image.fromarray(arr).save(d / f'{i:04d}.png')
    print('  Classifier images:', sum(DEMO_COUNTS.values()))

    # Segmentation: 200 random image-mask pairs
    Path(CFG['seg_img_dir']).mkdir(parents=True, exist_ok=True)
    Path(CFG['seg_mask_dir']).mkdir(parents=True, exist_ok=True)
    for i in range(200):
        arr = np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8)
        mask = np.zeros((256, 256), dtype=np.uint8)
        mask[50:200, 40:220] = 255   # synthetic lung region
        Image.fromarray(arr).save(Path(CFG['seg_img_dir']) / f'{i:04d}.png')
        Image.fromarray(mask).save(Path(CFG['seg_mask_dir']) / f'{i:04d}.png')
    print('  Segmentation pairs: 200')
    print('  Replace with real data before production training!')
else:
    print('Real data mode — verify paths in CFG')

## 4. Data Utilities — Split, Stats, Augmentation

In [ ]:
# ── 4.1  Collect file paths and labels ────────────────────────────────────────
from sklearn.model_selection import train_test_split

def collect_classifier_files(data_dir, classes):
    """Walk class sub-folders, return (paths, labels) lists."""
    paths, labels = [], []
    for idx, cls in enumerate(classes):
        cls_dir = Path(data_dir) / cls
        if not cls_dir.exists():
            print(f'  ⚠  Missing class folder: {cls_dir}')
            continue
        imgs = list(cls_dir.glob('*.png')) + list(cls_dir.glob('*.jpg')) \
             + list(cls_dir.glob('*.jpeg'))
        paths.extend(imgs)
        labels.extend([idx] * len(imgs))
    return paths, labels

all_paths, all_labels = collect_classifier_files(CFG['clf_data_dir'], CFG['classes'])

# Stratified train / val / test split
train_paths, tmp_paths, train_labels, tmp_labels = train_test_split(
    all_paths, all_labels,
    test_size=CFG['val_split'] + CFG['test_split'],
    stratify=all_labels, random_state=SEED
)
val_ratio = CFG['val_split'] / (CFG['val_split'] + CFG['test_split'])
val_paths, test_paths, val_labels, test_labels = train_test_split(
    tmp_paths, tmp_labels,
    test_size=1 - val_ratio,
    stratify=tmp_labels, random_state=SEED
)

print(f'Train : {len(train_paths):,} | Val : {len(val_paths):,} | Test : {len(test_paths):,}')
print('Train distribution:', {CFG["classes"][k]: v for k, v in Counter(train_labels).items()})

In [ ]:
# ── 4.2  Class imbalance visualization ────────────────────────────────────────
counts = [Counter(train_labels).get(i, 0) for i in range(CFG['n_classes'])]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(CFG['classes'], counts, color=plt.cm.Set2.colors[:5], edgecolor='black', linewidth=0.8)
for bar, c in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, str(c),
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Training Set Class Distribution', fontsize=13, fontweight='bold')
ax.set_ylabel('Sample Count')
plt.tight_layout()
plt.savefig('/content/class_distribution.png', dpi=150)
plt.show()

In [ ]:
# ── 4.3  Albumentations augmentation pipelines ───────────────────────────────
# Classifier — strong augmentation for training
clf_train_transform = A.Compose([
    A.Resize(CFG['clf_img_size'], CFG['clf_img_size']),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.15, rotate_limit=15, p=0.7),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3),
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8,8)),
        A.Equalize(),
    ], p=0.7),
    A.GaussNoise(var_limit=(10, 50), p=0.3),
    A.OneOf([
        A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50),
        A.GridDistortion(num_steps=5, distort_limit=0.3),
        A.OpticalDistortion(distort_limit=0.2, shift_limit=0.05),
    ], p=0.3),
    A.CoarseDropout(max_holes=8, max_height=20, max_width=20, fill_value=0, p=0.3),
    A.Normalize(mean=CFG['mean'], std=CFG['std']),
    ToTensorV2(),
])

clf_val_transform = A.Compose([
    A.Resize(CFG['clf_img_size'], CFG['clf_img_size']),
    A.Normalize(mean=CFG['mean'], std=CFG['std']),
    ToTensorV2(),
])

# TTA transform — slight variations for inference ensemble
clf_tta_transform = A.Compose([
    A.Resize(CFG['clf_img_size'], CFG['clf_img_size']),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.02, scale_limit=0.05, rotate_limit=5, p=0.5),
    A.Normalize(mean=CFG['mean'], std=CFG['std']),
    ToTensorV2(),
])

# Segmenter augmentation (spatial ops applied identically to image+mask)
seg_train_transform = A.Compose([
    A.Resize(CFG['seg_img_size'], CFG['seg_img_size']),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.6),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.GaussNoise(var_limit=(5, 30), p=0.3),
    A.ElasticTransform(alpha=1, sigma=30, alpha_affine=30, p=0.3),
    A.Normalize(mean=CFG['mean'], std=CFG['std']),
    ToTensorV2(),
])

seg_val_transform = A.Compose([
    A.Resize(CFG['seg_img_size'], CFG['seg_img_size']),
    A.Normalize(mean=CFG['mean'], std=CFG['std']),
    ToTensorV2(),
])

print('Augmentation pipelines created ✓')

## 5. Dataset Classes

In [ ]:
# ── 5.1  Classifier Dataset ───────────────────────────────────────────────────
class CXRClassifierDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths     = [str(p) for p in paths]
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.paths[idx])
        if img is None:
            # Return zeros if corrupt file
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform:
            img = self.transform(image=img)['image']

        return img, self.labels[idx]

# ── 5.2  Segmentation Dataset ─────────────────────────────────────────────────
class LungSegDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir   = Path(img_dir)
        self.mask_dir  = Path(mask_dir)
        self.transform = transform
        self.names     = sorted([
            p.name for p in self.img_dir.iterdir()
            if p.suffix.lower() in ('.png', '.jpg', '.jpeg')
        ])

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        name = self.names[idx]
        img  = cv2.imread(str(self.img_dir / name))
        mask_path = self.mask_dir / name
        if not mask_path.exists():
            # Try common mask naming conventions
            stem = Path(name).stem
            for ext in ('.png', '.jpg'):
                mp = self.mask_dir / (stem + ext)
                if mp.exists():
                    mask_path = mp; break

        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

        if img is None:  img  = np.zeros((256, 256, 3), dtype=np.uint8)
        if mask is None: mask = np.zeros((256, 256),    dtype=np.uint8)

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform:
            aug  = self.transform(image=img, mask=mask)
            img  = aug['image']                         # [3, H, W]
            mask = aug['mask'].unsqueeze(0).float() / 255.0  # [1, H, W] 0–1

        return img, mask

print('Dataset classes defined ✓')

In [ ]:
# ── 5.3  Weighted sampler to handle class imbalance ───────────────────────────
def build_weighted_sampler(labels):
    class_counts = Counter(labels)
    n_total      = len(labels)
    class_weights = {cls: n_total / count for cls, count in class_counts.items()}
    sample_weights = [class_weights[lbl] for lbl in labels]
    return WeightedRandomSampler(
        weights     = torch.DoubleTensor(sample_weights),
        num_samples = n_total,
        replacement = True
    )

# ── 5.4  Build DataLoaders ────────────────────────────────────────────────────
train_ds = CXRClassifierDataset(train_paths, train_labels, clf_train_transform)
val_ds   = CXRClassifierDataset(val_paths,   val_labels,   clf_val_transform)
test_ds  = CXRClassifierDataset(test_paths,  test_labels,  clf_val_transform)

sampler = build_weighted_sampler(train_labels)

clf_train_loader = DataLoader(
    train_ds, batch_size=CFG['clf_batch_size'],
    sampler=sampler, num_workers=CFG['num_workers'],
    pin_memory=True, drop_last=True
)
clf_val_loader = DataLoader(
    val_ds, batch_size=CFG['clf_batch_size'] * 2,
    shuffle=False, num_workers=CFG['num_workers'], pin_memory=True
)
clf_test_loader = DataLoader(
    test_ds, batch_size=CFG['clf_batch_size'] * 2,
    shuffle=False, num_workers=CFG['num_workers'], pin_memory=True
)

# Segmentation loaders
full_seg = LungSegDataset(CFG['seg_img_dir'], CFG['seg_mask_dir'])
seg_train_size = int(0.8 * len(full_seg))
seg_val_size   = len(full_seg) - seg_train_size
seg_train_ds, seg_val_ds = torch.utils.data.random_split(
    full_seg, [seg_train_size, seg_val_size],
    generator=torch.Generator().manual_seed(SEED)
)
seg_train_ds.dataset.transform = seg_train_transform
seg_val_ds.dataset.transform   = seg_val_transform

seg_train_loader = DataLoader(
    seg_train_ds, batch_size=CFG['seg_batch_size'],
    shuffle=True, num_workers=CFG['num_workers'], pin_memory=True, drop_last=True
)
seg_val_loader = DataLoader(
    seg_val_ds, batch_size=CFG['seg_batch_size'],
    shuffle=False, num_workers=CFG['num_workers'], pin_memory=True
)

print(f'Classifier  — train batches: {len(clf_train_loader)} | val: {len(clf_val_loader)}')
print(f'Segmenter   — train batches: {len(seg_train_loader)} | val: {len(seg_val_loader)}')

## 6. Model Architecture Definitions (Backend-Exact)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CLASSIFIER — DenseNet-121
#  Layer structure must remain identical to the backend class definition.
# ══════════════════════════════════════════════════════════════════════════════
def build_classifier(n_classes=5, pretrained=True):
    """Returns a DenseNet-121 with the backend-compatible classification head."""
    model = models.densenet121(weights='IMAGENET1K_V1' if pretrained else None)
    # ← This exact line must match the backend loader
    model.classifier = nn.Linear(1024, n_classes)
    return model

# Quick sanity check
_clf_test = build_classifier()
_x = torch.randn(2, 3, 224, 224)
_out = _clf_test(_x)
assert _out.shape == (2, 5), f'Classifier output mismatch: {_out.shape}'
print(f'Classifier output shape : {_out.shape}  ✓')
del _clf_test, _x, _out

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  SEGMENTER — Attention-UNet with scSE blocks
#  Architecture is reproduced verbatim from the backend specification.
# ══════════════════════════════════════════════════════════════════════════════

class SpatialSE(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv    = nn.Conv2d(in_channels, 1, kernel_size=1)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return x * self.sigmoid(self.conv(x))

class ChannelSE(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        flat = self.avg_pool(x).view(b, c)
        attention = self.fc(flat).view(b, c, 1, 1)
        return x * attention

class scSEBlock(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.cSE = ChannelSE(in_channels, reduction)
        self.sSE = SpatialSE(in_channels)
    def forward(self, x):
        return self.cSE(x) + self.sSE(x)

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            scSEBlock(out_channels)
        )
    def forward(self, x):
        return self.conv(x)

class AttentionUNet(nn.Module):
    def __init__(self, n_classes=1):
        super().__init__()
        self.enc1      = DoubleConv(3, 64)
        self.pool1     = nn.MaxPool2d(2)
        self.enc2      = DoubleConv(64, 128)
        self.pool2     = nn.MaxPool2d(2)
        self.enc3      = DoubleConv(128, 256)
        self.pool3     = nn.MaxPool2d(2)
        self.enc4      = DoubleConv(256, 512)
        self.pool4     = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(512, 1024)
        self.up4       = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4      = DoubleConv(1024, 512)
        self.up3       = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3      = DoubleConv(512, 256)
        self.up2       = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2      = DoubleConv(256, 128)
        self.up1       = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1      = DoubleConv(128, 64)
        self.final     = nn.Conv2d(64, n_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))
        b  = self.bottleneck(self.pool4(e4))
        d4 = self.up4(b)
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.dec4(d4)
        d3 = self.up3(d4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)
        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)
        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)
        return torch.sigmoid(self.final(d1))

# Shape verification
_seg = AttentionUNet(n_classes=1)
_x   = torch.randn(2, 3, 256, 256)
_out = _seg(_x)
assert _out.shape == (2, 1, 256, 256), f'Segmenter output mismatch: {_out.shape}'
print(f'Segmenter output shape  : {_out.shape}  ✓')
del _seg, _x, _out

## 7. Loss Functions & Training Utilities

In [ ]:
# ── 7.1  Focal Loss (handles extreme imbalance better than CE) ────────────────
class FocalLoss(nn.Module):
    """Focal Loss with optional per-class alpha weights."""
    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        super().__init__()
        self.gamma     = gamma
        self.alpha     = alpha      # tensor of shape [n_classes]
        self.reduction = reduction

    def forward(self, logits, targets):
        ce   = F.cross_entropy(logits, targets, weight=self.alpha, reduction='none')
        pt   = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean() if self.reduction == 'mean' else loss

# ── 7.2  Combined Dice + BCE loss for segmentation ───────────────────────────
class DiceBCELoss(nn.Module):
    def __init__(self, pos_weight=2.0, smooth=1e-5):
        super().__init__()
        self.bce    = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([pos_weight]))
        self.smooth = smooth

    def dice(self, pred_sigmoid, target):
        pred   = pred_sigmoid.view(-1)
        tgt    = target.view(-1)
        inter  = (pred * tgt).sum()
        return 1 - (2 * inter + self.smooth) / (pred.sum() + tgt.sum() + self.smooth)

    def forward(self, logits, targets):
        # Segmenter already applies sigmoid in forward(); we get probabilities
        # Use raw logits for BCEWithLogitsLoss to avoid double sigmoid
        # NOTE: AttentionUNet.forward() applies sigmoid — so logits here are probs.
        # We invert to logit space for numerical stability with BCEWithLogitsLoss.
        eps    = 1e-6
        raw_logits = torch.log((logits + eps) / (1 - logits + eps))
        bce_loss   = self.bce(raw_logits, targets)
        dice_loss  = self.dice(logits, targets)
        return 0.5 * bce_loss + 0.5 * dice_loss

# ── 7.3  Mixup / CutMix helpers ───────────────────────────────────────────────
def mixup_data(x, y, alpha=0.4):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    idx  = torch.randperm(x.size(0), device=x.device)
    xm   = lam * x + (1 - lam) * x[idx]
    ya, yb = y, y[idx]
    return xm, ya, yb, lam

def mixup_criterion(criterion, pred, ya, yb, lam):
    return lam * criterion(pred, ya) + (1 - lam) * criterion(pred, yb)

def cutmix_data(x, y, alpha=0.4):
    lam  = np.random.beta(alpha, alpha)
    idx  = torch.randperm(x.size(0), device=x.device)
    B, C, H, W = x.shape
    cx   = np.random.randint(W)
    cy   = np.random.randint(H)
    rw   = int(W * np.sqrt(1 - lam))
    rh   = int(H * np.sqrt(1 - lam))
    x1   = max(cx - rw // 2, 0);  x2 = min(cx + rw // 2, W)
    y1   = max(cy - rh // 2, 0);  y2 = min(cy + rh // 2, H)
    xc   = x.clone()
    xc[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam  = 1 - (x2 - x1) * (y2 - y1) / (W * H)
    return xc, y, y[idx], lam

# ── 7.4  Metrics ──────────────────────────────────────────────────────────────
def dice_score(pred, target, threshold=0.5, smooth=1e-5):
    pred   = (pred > threshold).float()
    inter  = (pred * target).sum()
    return ((2 * inter + smooth) / (pred.sum() + target.sum() + smooth)).item()

def iou_score(pred, target, threshold=0.5, smooth=1e-5):
    pred  = (pred > threshold).float()
    inter = (pred * target).sum()
    union = pred.sum() + target.sum() - inter
    return ((inter + smooth) / (union + smooth)).item()

print('Loss functions and utilities defined ✓')

In [ ]:
# ── 7.5  Compute class-frequency-based alpha weights ─────────────────────────
counts = np.array([Counter(train_labels).get(i, 1) for i in range(CFG['n_classes'])],
                  dtype=np.float32)
alpha_weights = torch.tensor((1.0 / counts) / (1.0 / counts).sum(),
                              device=DEVICE)
print('Focal loss alpha weights (per class):')
for cls, w in zip(CFG['classes'], alpha_weights):
    print(f'  {cls:<18}: {w:.4f}')

## 8. Classifier Training

In [ ]:
# ── 8.1  Training loop helpers ────────────────────────────────────────────────
class AverageMeter:
    def __init__(self): self.reset()
    def reset(self): self.val = self.avg = self.sum = self.count = 0
    def update(self, val, n=1):
        self.val    = val
        self.sum   += val * n
        self.count += n
        self.avg    = self.sum / self.count

class EarlyStopping:
    def __init__(self, patience=10, min_delta=1e-4, mode='max'):
        self.patience  = patience
        self.min_delta = min_delta
        self.mode      = mode
        self.best      = -np.inf if mode == 'max' else np.inf
        self.counter   = 0
        self.triggered = False

    def __call__(self, metric):
        improved = (metric > self.best + self.min_delta) if self.mode == 'max' \
                   else (metric < self.best - self.min_delta)
        if improved:
            self.best    = metric
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.triggered = True
        return self.triggered

In [ ]:
# ── 8.2  Train one epoch (classifier) ────────────────────────────────────────
def train_clf_epoch(model, loader, optimizer, scaler, criterion, epoch):
    model.train()
    loss_m  = AverageMeter()
    acc_m   = AverageMeter()
    optimizer.zero_grad()

    pbar = tqdm(loader, desc=f'Train E{epoch:03d}', leave=False)
    for step, (imgs, labels) in enumerate(pbar):
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        # Alternate between Mixup and CutMix
        r = random.random()
        if r < 0.35:
            imgs, ya, yb, lam = mixup_data(imgs, labels, CFG['clf_mixup_alpha'])
            mix_fn = lambda p: mixup_criterion(criterion, p, ya, yb, lam)
        elif r < 0.70:
            imgs, ya, yb, lam = cutmix_data(imgs, labels, CFG['clf_cutmix_alpha'])
            mix_fn = lambda p: mixup_criterion(criterion, p, ya, yb, lam)
        else:
            mix_fn = lambda p: criterion(p, labels)

        with autocast(enabled=CFG['amp']):
            logits = model(imgs)
            loss   = mix_fn(logits) / CFG['accum_steps']

        scaler.scale(loss).backward()

        if (step + 1) % CFG['accum_steps'] == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        acc = (logits.argmax(1) == labels).float().mean().item()
        loss_m.update(loss.item() * CFG['accum_steps'], imgs.size(0))
        acc_m.update(acc, imgs.size(0))
        pbar.set_postfix(loss=f'{loss_m.avg:.4f}', acc=f'{acc_m.avg:.4f}')

    return loss_m.avg, acc_m.avg

# ── 8.3  Validate (classifier) ────────────────────────────────────────────────
@torch.no_grad()
def eval_clf_epoch(model, loader):
    model.eval()
    all_preds, all_labels_list, all_probs = [], [], []

    for imgs, labels in loader:
        imgs   = imgs.to(DEVICE, non_blocking=True)
        with autocast(enabled=CFG['amp']):
            logits = model(imgs)
        probs  = torch.softmax(logits, dim=1).cpu()
        preds  = probs.argmax(1)
        all_preds.extend(preds.numpy())
        all_labels_list.extend(labels.numpy())
        all_probs.append(probs.numpy())

    all_probs = np.vstack(all_probs)
    acc       = accuracy_score(all_labels_list, all_preds)
    f1        = f1_score(all_labels_list, all_preds, average='macro')
    try:
        auc = roc_auc_score(all_labels_list, all_probs, multi_class='ovr', average='macro')
    except ValueError:
        auc = 0.0
    return acc, f1, auc, all_preds, all_labels_list

In [ ]:
# ── 8.4  Main classifier training loop ───────────────────────────────────────
clf_model = build_classifier(n_classes=CFG['n_classes'], pretrained=True).to(DEVICE)

# Criterion: Focal Loss with class-frequency alpha
clf_criterion = FocalLoss(gamma=2.0, alpha=alpha_weights)

# Optimizer: AdamW with differential LR (backbone lr = 0.1× head lr)
backbone_params = [p for n, p in clf_model.named_parameters() if 'classifier' not in n]
head_params     = [p for n, p in clf_model.named_parameters() if 'classifier'     in n]
clf_optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': CFG['clf_lr'] * 0.1},
    {'params': head_params,     'lr': CFG['clf_lr']},
], weight_decay=CFG['clf_weight_decay'])

# Scheduler: OneCycleLR for aggressive warmup + cosine decay
clf_scheduler = optim.lr_scheduler.OneCycleLR(
    clf_optimizer,
    max_lr        = [CFG['clf_lr'] * 0.1, CFG['clf_lr']],
    epochs        = CFG['clf_epochs'],
    steps_per_epoch = len(clf_train_loader),
    pct_start     = CFG['clf_warmup_epochs'] / CFG['clf_epochs'],
    div_factor    = 25,
    final_div_factor = 1e4,
)

scaler = GradScaler(enabled=CFG['amp'])
early_stop = EarlyStopping(patience=CFG['clf_patience'], mode='max')

clf_history = {'train_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': [], 'val_auc': []}
best_val_acc = 0.0

print('Starting classifier training...')
print(f"Model params: {sum(p.numel() for p in clf_model.parameters()):,}")

for epoch in range(1, CFG['clf_epochs'] + 1):
    tr_loss, tr_acc = train_clf_epoch(clf_model, clf_train_loader,
                                      clf_optimizer, scaler, clf_criterion, epoch)
    clf_scheduler.step()   # OneCycleLR steps per-batch; call once per epoch if batches already stepped

    val_acc, val_f1, val_auc, _, _ = eval_clf_epoch(clf_model, clf_val_loader)

    clf_history['train_loss'].append(tr_loss)
    clf_history['train_acc'].append(tr_acc)
    clf_history['val_acc'].append(val_acc)
    clf_history['val_f1'].append(val_f1)
    clf_history['val_auc'].append(val_auc)

    is_best = val_acc > best_val_acc
    if is_best:
        best_val_acc = val_acc
        torch.save(clf_model.state_dict(), CFG['clf_weights'])

    print(f'E{epoch:03d}  loss={tr_loss:.4f}  tr_acc={tr_acc:.4f}  '
          f'val_acc={val_acc:.4f}  val_f1={val_f1:.4f}  val_auc={val_auc:.4f}'
          + (' ← best' if is_best else ''))

    if early_stop(val_acc):
        print(f'Early stopping triggered at epoch {epoch}')
        break

    gc.collect()

print(f'\nBest val accuracy: {best_val_acc:.4f}')
print(f'Weights saved → {CFG["clf_weights"]}')

In [ ]:
# ── 8.5  Training curves ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(clf_history['train_loss'], label='Train Loss')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(clf_history['train_acc'], label='Train Acc')
axes[1].plot(clf_history['val_acc'],   label='Val Acc')
axes[1].axhline(0.97, color='red', linestyle='--', label='97% target')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()

axes[2].plot(clf_history['val_f1'],  label='Val F1 (macro)')
axes[2].plot(clf_history['val_auc'], label='Val AUC (macro)')
axes[2].set_title('F1 / AUC'); axes[2].set_xlabel('Epoch'); axes[2].legend()

plt.tight_layout()
plt.savefig('/content/clf_training_curves.png', dpi=150)
plt.show()

## 9. Segmenter Training

In [ ]:
# ── 9.1  Train one epoch (segmenter) ─────────────────────────────────────────
def train_seg_epoch(model, loader, optimizer, scaler, criterion, epoch):
    model.train()
    loss_m = AverageMeter()
    dice_m = AverageMeter()
    optimizer.zero_grad()

    pbar = tqdm(loader, desc=f'Seg Train E{epoch:03d}', leave=False)
    for step, (imgs, masks) in enumerate(pbar):
        imgs  = imgs.to(DEVICE,  non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)

        with autocast(enabled=CFG['amp']):
            preds = model(imgs)                    # already sigmoid-activated
            loss  = criterion(preds, masks) / CFG['accum_steps']

        scaler.scale(loss).backward()

        if (step + 1) % CFG['accum_steps'] == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        d = dice_score(preds.detach(), masks)
        loss_m.update(loss.item() * CFG['accum_steps'], imgs.size(0))
        dice_m.update(d, imgs.size(0))
        pbar.set_postfix(loss=f'{loss_m.avg:.4f}', dice=f'{dice_m.avg:.4f}')

    return loss_m.avg, dice_m.avg

@torch.no_grad()
def eval_seg_epoch(model, loader):
    model.eval()
    dice_m = AverageMeter()
    iou_m  = AverageMeter()

    for imgs, masks in loader:
        imgs  = imgs.to(DEVICE,  non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)
        with autocast(enabled=CFG['amp']):
            preds = model(imgs)
        dice_m.update(dice_score(preds, masks), imgs.size(0))
        iou_m.update(iou_score(preds, masks),   imgs.size(0))

    return dice_m.avg, iou_m.avg

In [ ]:
# ── 9.2  Main segmenter training loop ────────────────────────────────────────
seg_model     = AttentionUNet(n_classes=1).to(DEVICE)
seg_criterion = DiceBCELoss(pos_weight=CFG['seg_pos_weight']).to(DEVICE)
seg_optimizer = optim.AdamW(seg_model.parameters(),
                            lr=CFG['seg_lr'],
                            weight_decay=CFG['seg_weight_decay'])
seg_scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    seg_optimizer, T_0=10, T_mult=2, eta_min=1e-6
)

seg_scaler    = GradScaler(enabled=CFG['amp'])
seg_early_stop = EarlyStopping(patience=CFG['seg_patience'], mode='max')
seg_history   = {'train_loss': [], 'train_dice': [], 'val_dice': [], 'val_iou': []}
best_val_dice = 0.0

print('Starting segmenter training...')
print(f"Model params: {sum(p.numel() for p in seg_model.parameters()):,}")

for epoch in range(1, CFG['seg_epochs'] + 1):
    tr_loss, tr_dice = train_seg_epoch(seg_model, seg_train_loader,
                                       seg_optimizer, seg_scaler, seg_criterion, epoch)
    seg_scheduler.step()

    val_dice, val_iou = eval_seg_epoch(seg_model, seg_val_loader)

    seg_history['train_loss'].append(tr_loss)
    seg_history['train_dice'].append(tr_dice)
    seg_history['val_dice'].append(val_dice)
    seg_history['val_iou'].append(val_iou)

    is_best = val_dice > best_val_dice
    if is_best:
        best_val_dice = val_dice
        torch.save(seg_model.state_dict(), CFG['seg_weights'])

    print(f'E{epoch:03d}  loss={tr_loss:.4f}  tr_dice={tr_dice:.4f}  '
          f'val_dice={val_dice:.4f}  val_iou={val_iou:.4f}'
          + (' ← best' if is_best else ''))

    if seg_early_stop(val_dice):
        print(f'Early stopping triggered at epoch {epoch}')
        break

    gc.collect()

print(f'\nBest val Dice: {best_val_dice:.4f}')
print(f'Weights saved → {CFG["seg_weights"]}')

In [ ]:
# ── 9.3  Segmenter training curves ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(seg_history['train_loss'], label='Train Loss')
axes[0].set_title('Seg Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(seg_history['train_dice'], label='Train Dice')
axes[1].plot(seg_history['val_dice'],   label='Val Dice')
axes[1].plot(seg_history['val_iou'],    label='Val IoU')
axes[1].set_title('Dice / IoU'); axes[1].set_xlabel('Epoch'); axes[1].legend()

plt.tight_layout()
plt.savefig('/content/seg_training_curves.png', dpi=150)
plt.show()

## 10. Evaluation — Classifier

In [ ]:
# ── 10.1  Load best weights & evaluate on held-out test set ──────────────────
clf_model.load_state_dict(torch.load(CFG['clf_weights'], map_location=DEVICE))
test_acc, test_f1, test_auc, test_preds, test_true = eval_clf_epoch(clf_model, clf_test_loader)

print('=' * 55)
print('CLASSIFIER TEST SET RESULTS')
print('=' * 55)
print(f'Accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)')
print(f'F1 macro : {test_f1:.4f}')
print(f'AUC macro: {test_auc:.4f}')
print()
print(classification_report(test_true, test_preds, target_names=CFG['classes']))

In [ ]:
# ── 10.2  Confusion matrix ────────────────────────────────────────────────────
cm = confusion_matrix(test_true, test_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CFG['classes'], yticklabels=CFG['classes'],
            linewidths=0.5, ax=ax)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('True',      fontsize=11)
ax.set_title('Normalised Confusion Matrix (Test Set)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# ── 10.3  TTA evaluation (classifier) ────────────────────────────────────────
@torch.no_grad()
def eval_with_tta(model, paths, labels, n_aug=5):
    """Runs TTA over n_aug transforms and averages probabilities."""
    model.eval()
    all_probs = []

    for _ in range(n_aug):
        ds    = CXRClassifierDataset(paths, labels, clf_tta_transform)
        ldr   = DataLoader(ds, batch_size=CFG['clf_batch_size'] * 2,
                           shuffle=False, num_workers=CFG['num_workers'])
        run_probs = []
        for imgs, _ in ldr:
            imgs = imgs.to(DEVICE)
            with autocast(enabled=CFG['amp']):
                logits = model(imgs)
            run_probs.append(torch.softmax(logits, 1).cpu().numpy())
        all_probs.append(np.vstack(run_probs))

    avg_probs = np.mean(all_probs, axis=0)  # [N, n_classes]
    preds     = avg_probs.argmax(1)
    acc       = accuracy_score(labels, preds)
    f1        = f1_score(labels, preds, average='macro')
    return acc, f1, preds

tta_acc, tta_f1, tta_preds = eval_with_tta(
    clf_model, test_paths, test_labels, n_aug=CFG['clf_tta_n'])
print(f'TTA ({CFG["clf_tta_n"]}×) — Test Acc: {tta_acc:.4f}  F1: {tta_f1:.4f}')

## 11. Evaluation — Segmenter (Visual Samples)

In [ ]:
# ── 11.1  Load best seg weights & visualize ───────────────────────────────────
seg_model.load_state_dict(torch.load(CFG['seg_weights'], map_location=DEVICE))
seg_model.eval()

# Grab first 4 validation samples
batch_imgs, batch_masks = next(iter(seg_val_loader))
batch_imgs  = batch_imgs[:4].to(DEVICE)
batch_masks = batch_masks[:4]

with torch.no_grad():
    with autocast(enabled=CFG['amp']):
        preds = seg_model(batch_imgs).cpu()

mean_t = torch.tensor(CFG['mean']).view(3, 1, 1)
std_t  = torch.tensor(CFG['std']).view(3, 1, 1)

fig, axes = plt.subplots(4, 3, figsize=(10, 14))
for i in range(4):
    img_disp = (batch_imgs[i].cpu() * std_t + mean_t).permute(1, 2, 0).numpy().clip(0, 1)
    axes[i, 0].imshow(img_disp);                       axes[i, 0].set_title('Image')
    axes[i, 1].imshow(batch_masks[i, 0], cmap='gray'); axes[i, 1].set_title('GT Mask')
    axes[i, 2].imshow(preds[i, 0] > 0.5, cmap='gray');axes[i, 2].set_title('Pred Mask')
    for ax in axes[i]: ax.axis('off')

plt.suptitle('Segmentation Predictions (threshold=0.5)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('/content/seg_visual.png', dpi=150)
plt.show()

# Summary metrics
val_dice_final, val_iou_final = eval_seg_epoch(seg_model, seg_val_loader)
print(f'Final Val Dice: {val_dice_final:.4f}  |  IoU: {val_iou_final:.4f}')

## 12. Weight Export & Backend Compatibility Verification

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CRITICAL: Verify that saved weights load without errors into backend models
# ══════════════════════════════════════════════════════════════════════════════

def verify_weights(weight_path, model_fn, input_shape, expected_output_shape, label):
    """Loads weights into a fresh model and does a forward pass. Raises on mismatch."""
    model = model_fn().to(DEVICE)
    state = torch.load(weight_path, map_location=DEVICE)
    missing, unexpected = model.load_state_dict(state, strict=True)
    assert len(missing) == 0,    f'{label}: missing keys — {missing}'
    assert len(unexpected) == 0, f'{label}: unexpected keys — {unexpected}'

    model.eval()
    with torch.no_grad():
        dummy = torch.randn(input_shape, device=DEVICE)
        out   = model(dummy)
    assert tuple(out.shape) == expected_output_shape, \
        f'{label}: output shape {tuple(out.shape)} ≠ {expected_output_shape}'

    sz_mb = os.path.getsize(weight_path) / 1e6
    print(f'{label}')
    print(f'  File size     : {sz_mb:.1f} MB')
    print(f'  Load (strict) : ✓  (missing=0, unexpected=0)')
    print(f'  Output shape  : {tuple(out.shape)}  ✓')
    print()

print('=' * 55)
print('BACKEND COMPATIBILITY VERIFICATION')
print('=' * 55)

verify_weights(
    weight_path           = CFG['clf_weights'],
    model_fn              = lambda: build_classifier(n_classes=5, pretrained=False),
    input_shape           = (2, 3, 224, 224),
    expected_output_shape = (2, 5),
    label                 = 'custom_classifier.pth  (DenseNet-121, 5-class)'
)

verify_weights(
    weight_path           = CFG['seg_weights'],
    model_fn              = lambda: AttentionUNet(n_classes=1),
    input_shape           = (2, 3, 256, 256),
    expected_output_shape = (2, 1, 256, 256),
    label                 = 'custom_segmenter.pth   (Attention-UNet + scSE, binary)'
)

print('All verification checks passed ✓')

In [ ]:
# ── 12.1  Print layer summary for backend integration team ───────────────────
def print_layer_summary(model, label):
    print(f'\n── {label} ──')
    total = 0
    for name, param in model.named_parameters():
        total += param.numel()
    print(f'  Total parameters: {total:,}')
    # Print head details
    if hasattr(model, 'classifier'):
        print(f'  classifier layer : {model.classifier}')
    if hasattr(model, 'final'):
        print(f'  final layer      : {model.final}')

clf_fresh = build_classifier(n_classes=5, pretrained=False)
seg_fresh = AttentionUNet(n_classes=1)
print_layer_summary(clf_fresh, 'DenseNet-121 Classifier')
print_layer_summary(seg_fresh, 'Attention-UNet Segmenter')

## 13. Inference Smoke Test (End-to-End)

Simulates exactly what the backend does when loading the weights.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  INFERENCE SMOKE TEST
#  This cell replicates the backend inference path verbatim.
# ══════════════════════════════════════════════════════════════════════════════
import torchvision.transforms as T

CLASSES = ['Normal', 'Pneumonia', 'COVID-19', 'Tuberculosis', 'Lung Opacity']

# ── Classifier inference ──────────────────────────────────────────────────────
clf_infer = build_classifier(n_classes=5, pretrained=False).to(DEVICE)
clf_infer.load_state_dict(torch.load(CFG['clf_weights'], map_location=DEVICE))
clf_infer.eval()

clf_preprocess = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Synthetic test image (white noise)
from PIL import Image as PILImage
dummy_img = PILImage.fromarray(np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8))
input_tensor = clf_preprocess(dummy_img).unsqueeze(0).to(DEVICE)

with torch.no_grad():
    logits = clf_infer(input_tensor)
    probs  = torch.softmax(logits, dim=1)

pred_class = CLASSES[probs.argmax().item()]
print('Classifier smoke test')
print(f'  Input  : {tuple(input_tensor.shape)}')
print(f'  Output : {tuple(probs.shape)}')
print(f'  Prediction  : {pred_class}')
print(f'  Probabilities: {[f"{p:.3f}" for p in probs[0].cpu().tolist()]}')

# ── Segmenter inference ───────────────────────────────────────────────────────
seg_infer = AttentionUNet(n_classes=1).to(DEVICE)
seg_infer.load_state_dict(torch.load(CFG['seg_weights'], map_location=DEVICE))
seg_infer.eval()

seg_preprocess = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

seg_input = seg_preprocess(dummy_img).unsqueeze(0).to(DEVICE)

with torch.no_grad():
    mask_prob = seg_infer(seg_input)

print('\nSegmenter smoke test')
print(f'  Input  : {tuple(seg_input.shape)}')
print(f'  Output : {tuple(mask_prob.shape)}')
print(f'  Mask range    : [{mask_prob.min():.4f}, {mask_prob.max():.4f}]  ✓ (0–1 sigmoid)')

print('\n✅  All inference smoke tests passed')

## 14. Download Weights to Local Machine

In [ ]:
# ── Download directly from Colab ──────────────────────────────────────────────
try:
    from google.colab import files
    print('Downloading custom_classifier.pth ...')
    files.download(CFG['clf_weights'])
    print('Downloading custom_segmenter.pth ...')
    files.download(CFG['seg_weights'])
    print('Done!')
except ImportError:
    print('Not running in Colab — weights are at:')
    print(f'  {CFG["clf_weights"]}')
    print(f'  {CFG["seg_weights"]}')

In [ ]:
# ── Optional: upload to Google Drive ─────────────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy(CFG['clf_weights'], '/content/drive/MyDrive/model_weights/custom_classifier.pth')
# shutil.copy(CFG['seg_weights'], '/content/drive/MyDrive/model_weights/custom_segmenter.pth')
# print('Uploaded to Drive ✓')

---
## 📋 Checklist Before Production Training

| Step | Done? |
|------|-------|
| Replace demo data with real CXR datasets (NIH CheXpert / COVID-19 / TB) | ☐ |
| Set `DEMO_MODE = False` in Cell 3 | ☐ |
| Verify class folder names match `CFG['classes']` exactly | ☐ |
| Confirm segmentation mask format (0=background, 255=lung) | ☐ |
| Enable T4/A100 GPU in Colab runtime | ☐ |
| Adjust `clf_batch_size` / `seg_batch_size` to fit VRAM | ☐ |
| Run Cell 12 backend verification after training | ☐ |
| Confirm `custom_classifier.pth` loads with `strict=True` on backend | ☐ |
| Confirm `custom_segmenter.pth` loads with `strict=True` on backend | ☐ |

---
## 🔑 Key Design Decisions

| Decision | Rationale |
|----------|-----------|
| **Focal Loss + alpha weights** | Handles 8:1 class imbalance without synthetic oversampling artifacts |
| **WeightedRandomSampler** | Ensures underrepresented classes appear every batch |
| **Mixup + CutMix** | Improves calibration and reduces overconfidence on rare classes |
| **OneCycleLR** | Fast convergence with warm-up; outperforms fixed-LR in practice |
| **Differential LR** | Backbone trained at 10× lower LR to preserve ImageNet features |
| **Dice + BCE** | BCE handles class-level pixel imbalance; Dice maximises overlap |
| **AMP (float16)** | 2× throughput with no accuracy loss on T4/A100 |
| **Gradient accumulation** | Effective batch size = `batch_size × accum_steps` without more VRAM |
| **Strict weight loading** | `strict=True` in verification guarantees zero layer mismatch |
| **TTA at inference** | 5 augmented passes averaged → 0.5–1.5% accuracy uplift |